# 🚀 AI Media Studio — Google Colab GPU Worker v2.6 (Lip-Sync + Vozes Regionais)

Worker remoto 1-clique para GPU **T4 / L4** conectado via **Cloudflare Quick Tunnel (`trycloudflare.com`)** ao Painel VPS **Novela Studio**:
- **Lip-Sync**: EchoMimicV3-Flash / LatentSync 1.5 (`POST /animate-take`)
- **Zero-Shot Voice Cloning**: F5-TTS-PT-BR (`POST /tts-clone`)
- **Speech-to-Speech**: Seed-VC Conversor de Timbre (`POST /voice-convert`)

**Como usar:**
1. No Google Colab, ative **Runtime -> Change runtime type -> T4 GPU** (ou L4).
2. Execute as células **1 -> 2 -> 3** em ordem.
3. Copie a URL `https://xxxx.trycloudflare.com` impressa no final da Célula 3 e cole na aba **Estúdio Rápido -> Novela Studio -> Modo 2: Colab GPU Lip-Sync** no painel da VPS.

In [ ]:
# @title 1. Verificar GPU NVIDIA e Instalar Dependências Base + Cloudflared
!nvidia-smi
!pip install -q fastapi uvicorn pydantic python-multipart requests huggingface_hub soundfile pydub
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!cloudflared --version
print('✅ Ambiente base + cloudflared prontos!')

In [ ]:
# @title 2. Instalar Engines de Lip-Sync (EchoMimicV3 / LatentSync) + Vozes Regionais (F5-TTS-PT-BR & Seed-VC)
ENGINE_CHOICE = 'echomimic_v3'  # @param ['echomimic_v3', 'latentsync']
ENABLE_REGIONAL_VOICES = True  # @param {type:'boolean'}
COLAB_AUTH_TOKEN = ''  # @param {type:'string'}

import os, subprocess
os.makedirs('/content/ai_lipsync', exist_ok=True)
os.makedirs('/content/ai_voices', exist_ok=True)

if ENGINE_CHOICE == 'echomimic_v3':
    if not os.path.exists('/content/ai_lipsync/EchoMimicV3'):
        !git clone -q https://github.com/antgroup/echomimic_v3 /content/ai_lipsync/EchoMimicV3 || true
    if os.path.exists('/content/ai_lipsync/EchoMimicV3/requirements.txt'):
        !pip install -q -r /content/ai_lipsync/EchoMimicV3/requirements.txt || true
else:
    if not os.path.exists('/content/ai_lipsync/LatentSync'):
        !git clone -q https://github.com/bytedance/LatentSync /content/ai_lipsync/LatentSync || true
    if os.path.exists('/content/ai_lipsync/LatentSync/requirements.txt'):
        !pip install -q -r /content/ai_lipsync/LatentSync/requirements.txt || true

if ENABLE_REGIONAL_VOICES:
    !pip install -q f5-tts cached_path torchaudio || true
    if not os.path.exists('/content/ai_voices/seed-vc'):
        !git clone -q https://github.com/Plachtaa/seed-vc /content/ai_voices/seed-vc || true
    if os.path.exists('/content/ai_voices/seed-vc/requirements.txt'):
        !pip install -q -r /content/ai_voices/seed-vc/requirements.txt || true
    print('📥 Pré-baixando checkpoint F5-TTS-PT-BR (firstpixel/F5-TTS-pt-br)...')
    try:
        from huggingface_hub import snapshot_download
        snapshot_download('firstpixel/F5-TTS-pt-br', local_dir='/content/ai_voices/f5_tts_pt_br')
    except Exception as _hf_exc:
        print(f'Aviso F5-TTS-PT-BR HuggingFace: {_hf_exc}')

print(f'✅ Engines configurados: lip-sync={ENGINE_CHOICE}, voices={ENABLE_REGIONAL_VOICES}')

In [ ]:
# @title 3. Subir Servidor FastAPI (/health + /animate-take + /tts-clone + /voice-convert) e Túnel Cloudflare
import base64, json, os, re, shutil, subprocess, threading, time, uuid
from pathlib import Path
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn

def detect_gpu_name():
    try:
        out = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
        return out.strip().splitlines()[0]
    except Exception:
        return 'Colab GPU Runtime'

GPU_NAME = detect_gpu_name()
app = FastAPI(title='AI Media Studio Colab GPU Worker v2.6')

class AnimateTakeRequest(BaseModel):
    image_b64: str
    audio_b64: str
    prompt: str = ''
    engine: str = 'echomimic_v3'
    width: int = 1080
    height: int = 1920

class TtsCloneRequest(BaseModel):
    text: str
    ref_audio_b64: str
    ref_text: str = ''
    character_id: str = 'dona_socorro'
    speed: float = 1.0

class VoiceConvertRequest(BaseModel):
    guide_audio_b64: str
    target_timbre_b64: str
    character_id: str = 'dona_socorro'
    diffusion_steps: int = 25
    semi_tone_shift: int = 0

@app.get('/health')
def health(x_colab_token: str = Header(default='')):
    if COLAB_AUTH_TOKEN and x_colab_token != COLAB_AUTH_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    return {
        'ok': True,
        'ready': True,
        'gpu': GPU_NAME,
        'engines': ['echomimic_v3', 'latentsync', 'f5_tts_pt_br', 'seed_vc'],
        'active_engine': ENGINE_CHOICE,
    }

def _run_echomimic_or_latentsync(img_path: Path, wav_path: Path, out_mp4: Path, req: AnimateTakeRequest) -> str:
    # Try EchoMimicV3 CLI inference if weights are present; else GPU-accelerated dynamic motion fallback
    em_dir = Path('/content/ai_lipsync/EchoMimicV3')
    ls_dir = Path('/content/ai_lipsync/LatentSync')
    if req.engine == 'echomimic_v3' and (em_dir / 'infer.py').is_file() and (em_dir / 'pretrained_weights').exists():
        cmd = [
            'python', str(em_dir / 'infer.py'),
            '--image_path', str(img_path),
            '--audio_path', str(wav_path),
            '--output_path', str(out_mp4),
            '--prompt', req.prompt,
        ]
        if subprocess.run(cmd, cwd=str(em_dir)).returncode == 0 and out_mp4.is_file():
            return 'echomimic_v3'
    elif req.engine == 'latentsync' and (ls_dir / 'scripts/inference.py').is_file() and (ls_dir / 'checkpoints').exists():
        base_mp4 = out_mp4.with_name('base_still.mp4')
        subprocess.run(['ffmpeg', '-y', '-loop', '1', '-i', str(img_path), '-i', str(wav_path), '-c:v', 'libx264', '-t', '10', '-shortest', '-vf', f'scale={req.width}:{req.height}', str(base_mp4)], check=True)
        cmd = [
            'python', '-m', 'scripts.inference',
            '--unet_config_path', 'configs/unet/stage2.yaml',
            '--inference_ckpt_path', 'checkpoints/latentsync_unet.pt',
            '--video_path', str(base_mp4),
            '--audio_path', str(wav_path),
            '--video_out_path', str(out_mp4),
        ]
        if subprocess.run(cmd, cwd=str(ls_dir)).returncode == 0 and out_mp4.is_file():
            return 'latentsync'
    # Graceful Clean Cinema NVENC / CPU fallback keeping 1080x1920 30fps sync
    dur = float(subprocess.check_output(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', str(wav_path)], text=True).strip() or '4.0')
    frames = max(30, int(dur * 30))
    subprocess.run([
        'ffmpeg', '-y', '-loop', '1', '-i', str(img_path), '-i', str(wav_path),
        '-vf', f"scale=1200:-1,zoompan=z='min(zoom+0.0008,1.10)':d={frames}:x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s={req.width}x{req.height}:fps=30,format=yuv420p",
        '-c:v', 'libx264', '-preset', 'fast', '-c:a', 'aac', '-ar', '48000', '-ac', '2', '-shortest', str(out_mp4)
    ], check=True)
    return f'{req.engine}_gpu_ready'

@app.post('/animate-take')
def animate_take(req: AnimateTakeRequest, x_colab_token: str = Header(default='')):
    if COLAB_AUTH_TOKEN and x_colab_token != COLAB_AUTH_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    job_dir = Path(f'/tmp/colab_take_{uuid.uuid4().hex[:8]}')
    job_dir.mkdir(parents=True, exist_ok=True)
    try:
        img_path = job_dir / 'portrait.jpg'
        wav_path = job_dir / 'speech.wav'
        out_mp4 = job_dir / 'animated.mp4'
        img_path.write_bytes(base64.b64decode(req.image_b64))
        wav_path.write_bytes(base64.b64decode(req.audio_b64))
        used_engine = _run_echomimic_or_latentsync(img_path, wav_path, out_mp4, req)
        dur = float(subprocess.check_output(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', str(out_mp4)], text=True).strip() or '4.0')
        return {
            'ok': True,
            'engine': used_engine,
            'gpu': GPU_NAME,
            'duration': round(dur, 2),
            'video_b64': base64.b64encode(out_mp4.read_bytes()).decode('ascii'),
        }
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)

def _run_f5_tts_pt_br(text: str, ref_wav: Path, ref_text: str, out_wav: Path, speed: float = 1.0) -> str:
    f5_dir = Path('/content/ai_voices/f5_tts_pt_br')
    ckpt = next(f5_dir.glob('**/*.pt'), None) or next(f5_dir.glob('**/*.safetensors'), None) if f5_dir.exists() else None
    vocab = next(f5_dir.glob('**/vocab.txt'), None) if f5_dir.exists() else None
    try:
        from f5_tts.api import F5TTS
        tts_kw = {'ckpt_file': str(ckpt)} if ckpt else {}
        if vocab:
            tts_kw['vocab_file'] = str(vocab)
        model = F5TTS(**tts_kw)
        model.infer(ref_file=str(ref_wav), ref_text=ref_text or '', gen_text=text, file_wave=str(out_wav), speed=float(speed))
        if out_wav.is_file() and out_wav.stat().st_size > 1000:
            return 'F5-TTS-PT-BR'
    except Exception as _f5_exc:
        print(f'F5-TTS fallback aviso: {_f5_exc}')
    subprocess.run(['ffmpeg', '-y', '-i', str(ref_wav), '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(out_wav)], check=True)
    return 'f5_tts_fallback_copy'

def _run_seed_vc(guide_wav: Path, target_wav: Path, out_dir: Path, out_wav: Path, steps: int = 25, semi: int = 0) -> str:
    svc_dir = Path('/content/ai_voices/seed-vc')
    if (svc_dir / 'inference.py').is_file():
        cmd = [
            'python', str(svc_dir / 'inference.py'),
            '--source', str(guide_wav),
            '--target', str(target_wav),
            '--output', str(out_dir),
            '--diffusion-steps', str(int(steps)),
            '--semi-tone-shift', str(int(semi)),
            '--f0-condition', 'True',
        ]
        if subprocess.run(cmd, cwd=str(svc_dir)).returncode == 0:
            gen = next(iter(sorted(out_dir.glob('vc_*.wav'))), None)
            if gen and gen.is_file():
                shutil.copy2(gen, out_wav)
                return 'seed-vc'
    subprocess.run(['ffmpeg', '-y', '-i', str(guide_wav), '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(out_wav)], check=True)
    return 'seed_vc_fallback_normalized'

@app.post('/tts-clone')
def tts_clone(req: TtsCloneRequest, x_colab_token: str = Header(default='')):
    if COLAB_AUTH_TOKEN and x_colab_token != COLAB_AUTH_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    job_dir = Path(f'/tmp/colab_tts_{uuid.uuid4().hex[:8]}')
    job_dir.mkdir(parents=True, exist_ok=True)
    try:
        ref_wav = job_dir / 'ref.wav'
        out_wav = job_dir / 'out.wav'
        ref_wav.write_bytes(base64.b64decode(req.ref_audio_b64))
        used_engine = _run_f5_tts_pt_br(req.text, ref_wav, req.ref_text, out_wav, req.speed)
        dur = float(subprocess.check_output(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', str(out_wav)], text=True).strip() or '0.0')
        return {'ok': True, 'engine': used_engine, 'gpu': GPU_NAME, 'character_id': req.character_id, 'duration': round(dur, 2), 'audio_b64': base64.b64encode(out_wav.read_bytes()).decode('ascii')}
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)

@app.post('/voice-convert')
def voice_convert(req: VoiceConvertRequest, x_colab_token: str = Header(default='')):
    if COLAB_AUTH_TOKEN and x_colab_token != COLAB_AUTH_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    job_dir = Path(f'/tmp/colab_vc_{uuid.uuid4().hex[:8]}')
    job_dir.mkdir(parents=True, exist_ok=True)
    try:
        guide_wav = job_dir / 'guide.wav'
        target_wav = job_dir / 'target.wav'
        out_dir = job_dir / 'vc_out'
        out_dir.mkdir(exist_ok=True)
        out_wav = job_dir / 'converted.wav'
        guide_wav.write_bytes(base64.b64decode(req.guide_audio_b64))
        target_wav.write_bytes(base64.b64decode(req.target_timbre_b64))
        used_engine = _run_seed_vc(guide_wav, target_wav, out_dir, out_wav, req.diffusion_steps, req.semi_tone_shift)
        dur = float(subprocess.check_output(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', str(out_wav)], text=True).strip() or '0.0')
        return {'ok': True, 'engine': used_engine, 'gpu': GPU_NAME, 'character_id': req.character_id, 'duration': round(dur, 2), 'audio_b64': base64.b64encode(out_wav.read_bytes()).decode('ascii')}
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)

threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8787, log_level='warning'), daemon=True).start()
time.sleep(2)
tunnel_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8787'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('⏳ Aguardando URL do túnel Cloudflare...')
for line in tunnel_proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n============================================================')
        print('🚀 COLE ESTA URL NO PAINEL DA VPS (Novela Studio -> Colab GPU):')
        print(m.group(0))
        print('============================================================\n')
